<a href="https://colab.research.google.com/github/AmanuelDaget/Military-Aircraft-Recognition-Using-YOLO12/blob/main/Military_Aircraft_detection_from_Satellite_image_using_YOLO12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Military Aircraft Detection from Satellite image Using CNN and YOLO26: by Amanuel D**

**Install libraries**

In [1]:
!pip install ultralytics
!pip install lxml
!pip install seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.5 MB/s eta 0:00:00


**Import libraries**

In [2]:
import os
import os
import cv2
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xml.etree.ElementTree as ET

from tqdm import tqdm
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
import torch.optim as optim

from collections import Counter

from torchvision import transforms
from torchvision import datasets
from torchvision import models
from torch.utils.data import DataLoader

from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


**Mount Google Drive**

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


**Copy dataset from Drive**

In [4]:
!cp "/content/drive/MyDrive/Colab Notebooks/Computer Vision and Image Processing/Military_Aircraft_Detection_from_Satellite_image_CV_Project/MAR20.zip" /content/

Unzip the dataset
**bold text**

In [5]:
!unzip -q /content/MAR20.zip -d /content/MAR20

In [6]:
base_path = "/content/MAR20"

print(os.listdir(base_path))

['JPEGImages', 'ImageSets', 'Annotations']


**Define Dataset Paths**

In [7]:
BASE_DIR = "/content/MAR20"

IMAGE_DIR = os.path.join(BASE_DIR, "JPEGImages")

ANNOTATION_DIR = os.path.join(
    BASE_DIR,
    "Annotations",
    "Horizontal Bounding Boxes"
)

**`Read All Image IDs`**

In [8]:
all_image_ids = []

for file in os.listdir(ANNOTATION_DIR):

    if file.endswith('.xml'):

        image_id = file.replace('.xml', '')

        all_image_ids.append(image_id)

print("Total Images:", len(all_image_ids))

Total Images: 3842


**Create Clean Dataset Folders**

In [9]:
os.makedirs(
    '/content/cleaned_dataset/images',
    exist_ok=True
)

os.makedirs(
    '/content/cleaned_dataset/annotations',
    exist_ok=True
)

**Dataset Cleaning**

In [ ]:
valid_ids = []
bad_files = []

MIN_BOX_SIZE = 5

for image_id in tqdm(all_image_ids):

    image_path = os.path.join(
        IMAGE_DIR,
        image_id + '.jpg'
    )

    xml_path = os.path.join(
        ANNOTATION_DIR,
        image_id + '.xml'
    )

    # Check file existence
    if not os.path.exists(image_path):
        bad_files.append(image_id)
        continue

    if not os.path.exists(xml_path):
        bad_files.append(image_id)
        continue

    # Check image readability
    image = cv2.imread(image_path)

    if image is None:
        bad_files.append(image_id)
        continue

    # Validate XML
    try:

        root = ET.parse(xml_path).getroot()

        valid_annotation = True

        for obj in root.findall('object'):

            bbox = obj.find('bndbox')

            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)

            # Invalid boxes
            if xmin >= xmax or ymin >= ymax:
                valid_annotation = False
                break

            # Remove tiny boxes
            if (xmax - xmin) < MIN_BOX_SIZE:
                valid_annotation = False
                break

            if (ymax - ymin) < MIN_BOX_SIZE:
                valid_annotation = False
                break

        if valid_annotation:

            valid_ids.append(image_id)

            shutil.copy(
                image_path,
                '/content/cleaned_dataset/images'
            )

            shutil.copy(
                xml_path,
                '/content/cleaned_dataset/annotations'
            )

        else:
            bad_files.append(image_id)

    except:
        bad_files.append(image_id)

print("Valid Images:", len(valid_ids))
print("Bad Files:", len(set(bad_files)))

 23%|██▎       | 877/3842 [00:10<00:30, 96.22it/s] 

**Update Dataset Paths**

In [ ]:
IMAGE_DIR = '/content/cleaned_dataset/images'

ANNOTATION_DIR = '/content/cleaned_dataset/annotations'

Train/Val/Test Split

In [ ]:
train_ids, temp_ids = train_test_split(
    valid_ids,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Training Images :", len(train_ids))
print("Validation Images:", len(val_ids))
print("Test Images (Unseen):", len(test_ids))

**Analyze Class Distribution**

In [ ]:
class_counts = Counter()

for xml_file in os.listdir(ANNOTATION_DIR):

    xml_path = os.path.join(
        ANNOTATION_DIR,
        xml_file
    )

    root = ET.parse(xml_path).getroot()

    for obj in root.findall('object'):

        class_name = obj.find('name').text

        class_counts[class_name] += 1

print(class_counts)

**Plot Class Distribution**

In [ ]:
classes = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(15,6))

plt.bar(classes, counts)

plt.xticks(rotation=45)

plt.xlabel('Aircraft Class')
plt.ylabel('Instances')
plt.title('Class Distribution')

plt.show()

**Visualize Random Samples**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Sample Images from Each Split", fontsize=14, fontweight='bold')

for ax, (split_name, split_ids) in zip(
    axes,
    [("Train", train_ids), ("Validation", val_ids), ("Test", test_ids)]
):
    sample_id = random.choice(split_ids)
    image_path = os.path.join(IMAGE_DIR, sample_id + '.jpg')
    xml_path = os.path.join(ANNOTATION_DIR, sample_id + '.xml')

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    root = ET.parse(xml_path).getroot()
    for obj in root.findall('object'):
        class_name = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)
        cv2.putText(image, class_name, (xmin, ymin - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    ax.imshow(image)
    ax.set_title(f"{split_name} Sample")
    ax.axis('off')

plt.tight_layout()
plt.show()

**Create YOLO Dataset Structure**

In [ ]:
os.makedirs('/content/yolo_dataset/images/train', exist_ok=True)
os.makedirs('/content/yolo_dataset/images/val', exist_ok=True)
os.makedirs('/content/yolo_dataset/images/test', exist_ok=True)

os.makedirs('/content/yolo_dataset/labels/train', exist_ok=True)
os.makedirs('/content/yolo_dataset/labels/val', exist_ok=True)
os.makedirs('/content/yolo_dataset/labels/test', exist_ok=True)

**Create Class Mapping**

In [ ]:
classes = sorted(list(class_counts.keys()))

class_to_id = {
    cls:i for i, cls in enumerate(classes)
}

id_to_class = {
    i:cls for cls, i in class_to_id.items()
}

print(class_to_id)

**Convert XML to YOLO Format**

In [ ]:
def convert_to_yolo(size, box):

    w, h = size

    xmin, ymin, xmax, ymax = box

    x_center = ((xmin + xmax) / 2) / w
    y_center = ((ymin + ymax) / 2) / h

    width = (xmax - xmin) / w
    height = (ymax - ymin) / h

    return (
        x_center,
        y_center,
        width,
        height
    )

Convert Entire Dataset

In [ ]:
def process_dataset(image_ids, split='train'):

    for image_id in tqdm(image_ids):

        image_file = image_id + '.jpg'
        xml_file = image_id + '.xml'

        image_path = os.path.join(
            IMAGE_DIR,
            image_file
        )

        xml_path = os.path.join(
            ANNOTATION_DIR,
            xml_file
        )

        image = cv2.imread(image_path)

        h, w, _ = image.shape

        label_lines = []

        root = ET.parse(xml_path).getroot()

        for obj in root.findall('object'):

            class_name = obj.find('name').text

            class_id = class_to_id[class_name]

            bbox = obj.find('bndbox')

            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)

            yolo_box = convert_to_yolo(
                (w,h),
                (xmin,ymin,xmax,ymax)
            )

            label_line = (
                f"{class_id} " +
                " ".join(map(str, yolo_box))
            )

            label_lines.append(label_line)

        shutil.copy(
            image_path,
            f'/content/yolo_dataset/images/{split}/{image_file}'
        )

        with open(
            f'/content/yolo_dataset/labels/{split}/{image_id}.txt',
            'w'
        ) as f:

            f.write('\n'.join(label_lines))

**Process Train and Validation Sets**

In [ ]:
process_dataset(train_ids, split='train')
process_dataset(val_ids, split='val')
process_dataset(test_ids, split='test')

**Create YOLO YAML File**

In [ ]:
yaml_content = f"""
path: /content/yolo_dataset

train: images/train
val: images/val
test: images/test

names:
"""
for i, cls in enumerate(classes):
    yaml_content += f'  {i}: {cls}\n'

with open('/content/yolo_dataset/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print("YAML saved:")
print(yaml_content)

Load YOLO12 Model

In [ ]:
model = YOLO('yolo12n.pt')

**Train YOLO12**

In [ ]:
model.train(
    data='/content/yolo_dataset/dataset.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=10,
    optimizer='AdamW',
    lr0=0.001,
    pretrained=True,
    cache=True,
    amp=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2
)

**Load Best Trained Model**

In [ ]:
# Load best trained model
model = YOLO("/content/runs/detect/train/weights/best.pt")

** Evaluate on Validation Set**

In [ ]:
print("=== Validation Set Evaluation ===")
val_metrics = model.val(
    data='/content/yolo_dataset/dataset.yaml',
    split='val'
)
print(val_metrics)

**Evaluate on Unseen Test Set**

In [ ]:
print("=== Test Set Evaluation (Unseen Data) ===")
test_metrics = model.val(
    data='/content/yolo_dataset/dataset.yaml',
    split='test'
)
print(test_metrics)

**Run Prediction**

In [ ]:
results = model.predict(
    source='/content/yolo_dataset/images/test',
    conf=0.25,
    save=True
)

**Display Prediction Result**

In [ ]:
prediction_path = '/content/runs/detect/predict'

files = os.listdir(prediction_path)

sample_image = os.path.join(
    prediction_path,
    files[0]
)

img = Image.open(sample_image)

plt.figure(figsize=(5,5))

plt.imshow(img)

plt.axis('off')

plt.show()

**Locate YOLO Training Results Folder**

In [ ]:
import os

runs_dir = "/content/runs/detect"

print(os.listdir(runs_dir))

**Load Training Results**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_path = "/content/runs/detect/train/results.csv"

df = pd.read_csv(results_path)

df.head()

**All Training Plots Combined**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("YOLO12 Training Results", fontsize=16, fontweight='bold')

# Training Loss
axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Box Loss')
axes[0, 0].plot(df['epoch'], df['train/cls_loss'], label='Class Loss')
axes[0, 0].plot(df['epoch'], df['train/dfl_loss'], label='DFL Loss')
axes[0, 0].set_title("Training Loss Curves")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid()

# Validation Loss
axes[0, 1].plot(df['epoch'], df['val/box_loss'], label='Box Loss', linestyle='--')
axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Class Loss', linestyle='--')
axes[0, 1].plot(df['epoch'], df['val/dfl_loss'], label='DFL Loss', linestyle='--')
axes[0, 1].set_title("Validation Loss Curves")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].legend()
axes[0, 1].grid()

# mAP
axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50', color='green')
axes[1, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95', color='orange')
axes[1, 0].set_title("mAP Accuracy Curve")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("mAP")
axes[1, 0].legend()
axes[1, 0].grid()

# Precision & Recall
axes[1, 1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='blue')
axes[1, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='red')
axes[1, 1].set_title("Precision vs Recall")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Score")
axes[1, 1].legend()
axes[1, 1].grid()

plt.tight_layout()
plt.savefig("training_results.png", dpi=150)
plt.show()

**Confusion Matrix**

In [ ]:
conf_matrix_path = "/content/runs/detect/train/confusion_matrix.png"
import cv2
import matplotlib.pyplot as plt

img = cv2.imread(conf_matrix_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,10))
plt.imshow(img)
plt.axis("off")
plt.title("Confusion Matrix")
plt.show()

**Confusion Matrix (Normalized Version)**

In [ ]:
conf_matrix_norm_path = "/content/runs/detect/train/confusion_matrix_normalized.png"

img = cv2.imread(conf_matrix_norm_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,10))
plt.imshow(img)
plt.axis("off")
plt.title("Normalized Confusion Matrix")
plt.show()

**Val vs Test Comparison**

In [ ]:
print(f"{'Metric':<20} {'Validation':>12} {'Test (Unseen)':>14}")
print("-" * 48)
print(f"{'mAP@50':<20} {val_metrics.box.map50:>12.4f} {test_metrics.box.map50:>14.4f}")
print(f"{'mAP@50-95':<20} {val_metrics.box.map:>12.4f} {test_metrics.box.map:>14.4f}")
print(f"{'Precision':<20} {val_metrics.box.mp:>12.4f} {test_metrics.box.mp:>14.4f}")
print(f"{'Recall':<20} {val_metrics.box.mr:>12.4f} {test_metrics.box.mr:>14.4f}")

**Model Performance Summary**

In [ ]:
best_row = df.loc[df['metrics/mAP50(B)'].idxmax()]

print("=== Best Epoch (from Training) ===")
print("Best Epoch    :", int(best_row['epoch']))
print("mAP@50        :", round(best_row['metrics/mAP50(B)'], 4))
print("mAP@50-95     :", round(best_row['metrics/mAP50-95(B)'], 4))
print("Precision     :", round(best_row['metrics/precision(B)'], 4))
print("Recall        :", round(best_row['metrics/recall(B)'], 4))

print("\n=== Final Test Set (Unseen Data) ===")
print("mAP@50        :", round(test_metrics.box.map50, 4))
print("mAP@50-95     :", round(test_metrics.box.map, 4))
print("Precision     :", round(test_metrics.box.mp, 4))
print("Recall        :", round(test_metrics.box.mr, 4))